# ANN Code Illustration

This notebook gives small code examples for `01_Introduction_to_ANN.ipynb`.

It includes:

- Tensor shapes
- A simple MLP
- Forward pass
- Loss calculation
- Backpropagation
- Training loop on synthetic data


In [ ]:
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)


## Synthetic Binary Classification Data


In [ ]:
n_samples = 600
n_features = 2

X = torch.randn(n_samples, n_features)
true_w = torch.tensor([[2.0], [-3.0]])
true_b = torch.tensor([0.5])

logits = X @ true_w + true_b
y = (torch.sigmoid(logits) > 0.5).float()

dataset = TensorDataset(X, y)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

X.shape, y.shape


## A Small MLP


In [ ]:
class SimpleANN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 8),
            nn.ReLU(),
            nn.Linear(8, 4),
            nn.ReLU(),
            nn.Linear(4, 1),
        )

    def forward(self, x):
        return self.net(x)

model = SimpleANN()
model


## Forward Pass and Loss


In [ ]:
loss_fn = nn.BCEWithLogitsLoss()

sample_x, sample_y = next(iter(loader))
sample_logits = model(sample_x)
loss = loss_fn(sample_logits, sample_y)

print('logits shape:', sample_logits.shape)
print('loss:', loss.item())


## Backpropagation and One Update


In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

optimizer.zero_grad()
loss.backward()
optimizer.step()

print('One parameter update completed.')


## Full Training Loop


In [ ]:
model = SimpleANN()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.BCEWithLogitsLoss()

for epoch in range(20):
    total_loss = 0.0
    for xb, yb in loader:
        logits = model(xb)
        loss = loss_fn(logits, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * xb.size(0)

    avg_loss = total_loss / len(dataset)
    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch + 1:02d} | loss = {avg_loss:.4f}')


## Accuracy


In [ ]:
with torch.no_grad():
    probs = torch.sigmoid(model(X))
    preds = (probs > 0.5).float()
    accuracy = (preds == y).float().mean()

print(f'Accuracy: {accuracy.item():.3f}')
